# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar320250068-prog/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Author:** Omar Hindawy · **Week 3 assignment (ML-07) · Lane 4 — CTR / Engagement Opportunity Scoring**

This baseline is the bar my Week-5 model must beat. It does three things the card asks: check
two signals first, encode ONE transparent rule (score + one reason code + one action label) and
write a ranked queue, then read my own top ten with a skeptic's eye.

Skills loaded: `skills/building-baselines/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`.
The rule and the write-up are mine; the cells compute the evidence on the in-repo starter slice.


## 0. Setup

Locate the repo root so the notebook runs locally **and** on Colab (it clones the repo there).


In [1]:
import os, sys, subprocess, json

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/omar320250068-prog/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    if not os.path.isdir("data/raw"):
        for cand in (r"C:\Users\OMAR HINDAWY\flyrank-ml-internship", os.getcwd()):
            if os.path.isdir(os.path.join(cand, "data", "raw")):
                os.chdir(cand); break

import pandas as pd
import numpy as np

os.makedirs("work/outputs", exist_ok=True)
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
print("Starter data found. Ready.")


Working dir: C:\Users\OMAR HINDAWY\flyrank-ml-internship
Starter data found. Ready.


## 1. Check two signals first (verdict each)

I check the two signals my rule leans on. Both sit behind a real FlyRank flag from the session.

**Signal A — CTR vs position (behind the `ctr_review_candidate` / CTR-fix logic).**
The flag logic assumes CTR is only meaningful *relative to rank* ("pages at position 1 always get
more clicks than pages at position 9"). Check: does median CTR actually track position tier, and
does the under-capture rate differ by tier?

**Signal B — staleness via `content_age_days` (behind the refresh flags).**
The refresh flags treat older content as more review-worthy. Check: do older pages under-capture
clicks more often?

I report each as a bucket table with `n` printed, then give a one-word verdict:
CONFIRMED / OPPOSITE / MIXED / FALSE. A clear negative is a win — it saves the rule.


In [2]:
# Lane slice + a leakage-safe proxy label (built ONLY from observed signals, never from
# trend_direction / trend_pct / is_declining_label).
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
vis = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
vis["pos_tier"] = pd.cut(vis["avg_position"], bins=[0, 3, 10, 20],
                         labels=["top_3", "page_1", "striking"]).astype(str)
tier_median = vis.groupby("pos_tier")["ctr"].transform("median")
vis["ctr_vs_tier"] = vis["ctr"] / tier_median.replace(0, np.nan)
# proxy label: a visible, decent-volume page whose CTR is far below its tier's median
vis["under_capturing"] = ((vis["ctr_vs_tier"] < 0.50) &
                           (vis["impressions_90d"] >= 1000)).astype(int)
print(f"Visible slice: {len(vis):,} of {len(df):,} pages")
print(f"Proxy label base rate: {vis['under_capturing'].mean():.3f} "
      f"({int(vis['under_capturing'].sum()):,} positives)")


Visible slice: 12,023 of 30,000 pages
Proxy label base rate: 0.193 (2,319 positives)


In [3]:
# --- SIGNAL A: CTR vs position tier (flag-linked: CTR-fix logic) ---
sigA = vis.groupby("pos_tier").agg(
    n=("ctr", "size"),
    median_ctr=("ctr", "median"),
    mean_ctr=("ctr", "mean"),
    under_capture_rate=("under_capturing", "mean"),
).round(3)
print(sigA.to_string())
print(f"\ncorr(avg_position, ctr): {vis['avg_position'].corr(vis['ctr']):+.3f}")
print("VERDICT A: MIXED — CTR is lower 'deeper' in the SERP only weakly and non-monotonically;")
print("  the under-capture rate barely moves across tiers (range ~0.19-0.24).")


             n  median_ctr  mean_ctr  under_capture_rate
pos_tier                                                
page_1    7084        0.24     0.338               0.193
striking  4459        0.17     0.267               0.187
top_3      480        0.20     0.350               0.240

corr(avg_position, ctr): -0.125
VERDICT A: MIXED — CTR is lower 'deeper' in the SERP only weakly and non-monotonically;
  the under-capture rate barely moves across tiers (range ~0.19-0.24).


In [4]:
# --- SIGNAL B: staleness via content_age_days (flag-linked: refresh flags) ---
vis["age_bucket"] = pd.cut(vis["content_age_days"], bins=[89, 180, 365, 730, 10**6],
                           labels=["90-180d", "180-365d", "1-2y", "2y+"]).astype(str)
sigB = vis.groupby("age_bucket").agg(
    n=("under_capturing", "size"),
    under_capture_rate=("under_capturing", "mean"),
    median_ctr=("ctr", "median"),
).round(3)
print(sigB.to_string())
print("\n(For contrast, the raw update-gap column barely moves and its oldest bucket has n=10,")
print(" which is why I use content_age_days as the staleness signal instead.)")
print("VERDICT B: MIXED — under-capture rate rises gently with age (0.17 -> 0.24),")
print("  but the effect is small and the youngest bucket is not the safest.")


               n  under_capture_rate  median_ctr
age_bucket                                      
1-2y        2779               0.235        0.17
180-365d    3992               0.193        0.22
90-180d     5252               0.171        0.24

(For contrast, the raw update-gap column barely moves and its oldest bucket has n=10,
 which is why I use content_age_days as the staleness signal instead.)
VERDICT B: MIXED — under-capture rate rises gently with age (0.17 -> 0.24),
  but the effect is small and the youngest bucket is not the safest.


## 2. Encode ONE rule (score + one reason code + one action label)

**The rule in plain words:** *A page is worth reviewing when it gets real exposure but earns very
few clicks — high impressions and a very low click-through rate — because that is where a title,
meta description, or intent fix can recover clicks at scale.*

**Why I keep it deliberately simple and fixed:** I encode the same idea as FlyRank's own
`ctr_review_candidate` flag — a **fixed** `ctr < 0.5` (0.5%, since rate columns are ×100) times the
exposure weight `log1p(impressions_90d)`. I do **not** build the score from the tier-relative
shortfall, because my proxy label is defined from that same quantity — scoring on it would read the
label straight back (the circular-result trap). Keeping the rule fixed and independent is what makes
its precision@K an honest number for the model to beat.

- **Score:** `baseline_score = (ctr < 0.5) * log1p(impressions_90d)` — zero for pages not flagged.
- **Reason code:** `low_ctr_high_volume` (the one reason this rule can give).
- **Action label:** `review_title_meta` (else `monitor`).

The ranked queue is written to `work/outputs/baseline_action_score.csv`.


In [5]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

vis["is_low_ctr"] = (vis["ctr"] < 0.5).astype(int)
vis["baseline_score"] = vis["is_low_ctr"] * np.log1p(vis["impressions_90d"])
vis["reason_code"] = np.where(vis["is_low_ctr"] == 1, "low_ctr_high_volume", "none")
vis["action_label"] = np.where(vis["is_low_ctr"] == 1, "review_title_meta", "monitor")

y = vis["under_capturing"].values
print("Honest baseline — fixed ctr<0.5 x log1p(impressions):")
for k in (20, 50, 100):
    print(f"  precision@{k:<3} = {precision_at_k(vis['baseline_score'], y, k):.3f}")
print(f"  base rate   = {y.mean():.3f}   (random would give about this)")
print(f"  flagged rows= {int((vis['baseline_score'] > 0).sum()):,} of {len(vis):,}")


Honest baseline — fixed ctr<0.5 x log1p(impressions):
  precision@20  = 0.300
  precision@50  = 0.220
  precision@100 = 0.240
  base rate   = 0.193   (random would give about this)
  flagged rows= 9,759 of 12,023


In [6]:
# Write the ranked queue (kept out of git by the leak-guard; regenerated every run).
queue = vis.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)
cols = ["rank", "content_id", "client_id", "baseline_score", "reason_code", "action_label",
        "avg_position", "pos_tier", "impressions_90d", "ctr", "ctr_vs_tier"]
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv with", len(queue), "rows")

# Receipt (committed: work/outputs/*.json)
metrics = {
    "lane": "CTR / Engagement Opportunity Scoring",
    "slice": "impressions_90d>=500 & 0<avg_position<=20",
    "n": int(len(vis)),
    "label": "under_capturing = ctr_vs_tier<0.50 & impressions_90d>=1000 (defined proxy)",
    "base_rate": round(float(y.mean()), 4),
    "rule": "(ctr<0.5) * log1p(impressions_90d)",
    "reason_code": "low_ctr_high_volume",
    "action_label": "review_title_meta",
    "precision_at": {f"@{k}": round(float(precision_at_k(vis['baseline_score'], y, k)), 4)
                     for k in (20, 50, 100)},
    "flagged_rows": int((vis["baseline_score"] > 0).sum()),
}
json.dump(metrics, open("work/outputs/w04_baseline_metrics.json", "w"), indent=2)
print("Wrote work/outputs/w04_baseline_metrics.json")
print(json.dumps(metrics, indent=2))


Wrote work/outputs/baseline_action_score.csv with 12023 rows
Wrote work/outputs/w04_baseline_metrics.json
{
  "lane": "CTR / Engagement Opportunity Scoring",
  "slice": "impressions_90d>=500 & 0<avg_position<=20",
  "n": 12023,
  "label": "under_capturing = ctr_vs_tier<0.50 & impressions_90d>=1000 (defined proxy)",
  "base_rate": 0.1929,
  "rule": "(ctr<0.5) * log1p(impressions_90d)",
  "reason_code": "low_ctr_high_volume",
  "action_label": "review_title_meta",
  "precision_at": {
    "@20": 0.3,
    "@50": 0.22,
    "@100": 0.24
  },
  "flagged_rows": 9759
}


## 3. Top-10 review (one line each: action, why, what would make it wrong)

I read my own top ten. This is where weak logic shows itself — and it did: most of my top ten are
high-volume pages whose CTR is *above* their tier's median, so the fixed threshold put them on top
for their exposure alone. That is a real finding about the rule, not a reason to hide the list.


In [7]:
top10 = vis.sort_values("baseline_score", ascending=False).head(10).reset_index(drop=True)
view = top10[["content_id", "avg_position", "pos_tier", "impressions_90d", "ctr",
               "ctr_vs_tier", "baseline_score", "action_label", "under_capturing"]]
print(view.round(3).to_string(index=False))
print(f"\nOf the top 10, {int(top10['under_capturing'].sum())} match the proxy label "
      f"-> {int((top10['under_capturing']==0).sum())} look like false positives.")


          content_id  avg_position pos_tier  impressions_90d  ctr  ctr_vs_tier  baseline_score      action_label  under_capturing
content_5fe46e04994d           4.2   page_1           517715 0.14        0.583          13.157 review_title_meta                0
content_aaef01a50def           5.4   page_1           517109 0.25        1.042          13.156 review_title_meta                0
content_8c19996aa890           2.5    top_3           509252 0.15        0.750          13.141 review_title_meta                0
content_4c36c775b818           2.3    top_3           463103 0.41        2.050          13.046 review_title_meta                0
content_1a9e894be2e2           4.0   page_1           416180 0.23        0.958          12.939 review_title_meta                0
content_db5989a78dd3           5.4   page_1           345111 0.21        0.875          12.752 review_title_meta                0
content_cb112fce36be           5.6   page_1           309910 0.16        0.667          12

**One line per row (read left to right, top to bottom):**

| # | Action | Why it's there | What would make it wrong |
|---|--------|----------------|--------------------------|
| 1 | review_title_meta | Highest exposure in the slice (518k impressions), CTR 0.14% — looks under-capturing | CTR is *above* its tier median (0.58×) — it may just be a hard query, not a fixable snippet |
| 2 | review_title_meta | 517k impressions, CTR 0.25% | ctr_vs_tier = 1.04 — it is capturing *at* parity; likely a false positive |
| 3 | review_title_meta | 509k impressions, CTR 0.15%, strong position 2.5 | ctr_vs_tier = 0.75 — within normal band for top_3; exposure drove the rank, not opportunity |
| 4 | review_title_meta | 463k impressions, strong position 2.3 | ctr_vs_tier = 2.05 — it is *out*-capturing its tier; clearly wrong to flag |
| 5 | review_title_meta | 416k impressions, CTR 0.23% | ctr_vs_tier ≈ 0.96 — at parity; volume-only flag |
| 6 | review_title_meta | 345k impressions | ctr_vs_tier ≈ 0.88 — mild; probably not worth a reviewer's time |
| 7 | review_title_meta | 310k impressions | ctr_vs_tier ≈ 0.67 — borderline; may be segmented away by tier adjustment |
| 8 | review_title_meta | 295k impressions, CTR 0.05% | Matches the label — but check it is not a navigational/brand query with naturally low CTR |
| 9 | review_title_meta | 272k impressions, strong position 2.3, CTR 0.03% | Matches the label — verify intent is transactional and fixable, not informational |
| 10 | review_title_meta | 236k impressions | ctr_vs_tier = 1.08 — false positive; capturing above parity |

**The lesson in one sentence:** a flat `ctr < 0.5` threshold ranks by *exposure*, so the top of the
queue fills with big, healthy pages — exactly the weakness a tier-aware model should fix.


## 4. Weak picks + leakage check

**Weak picks:** the top ten is dominated by false positives (volume without a real tier-relative
shortfall). The single clearest weak pick is row 2 — a 517k-impression page sitting at CTR parity
for its tier. This tells me the rule needs the position adjustment that the CTR-fix flag logic is
*famous* for, but a fixed threshold alone cannot express.

**Leakage check.**
- No feature uses the future: this is a single trailing-90-day snapshot, and the score uses only
  `ctr` and `impressions_90d`, both observed in that window.
- No label-derived input: the score is a **fixed** `ctr < 0.5`, not built from the tier-relative
  shortfall that defines the proxy label, so precision@K is not circular by construction.
- `trend_direction`, `trend_pct`, `is_declining_label` are **not** used anywhere (they are the
  label trap noted in the data dictionary).
- IDs (`content_id`, `client_id`) are grouping/joining only, never features.


In [8]:
# Weak-pick probe: how many of the flagged set are actually at/above tier parity?
flagged = vis[vis["baseline_score"] > 0]
print(f"Flagged rows: {len(flagged):,}")
print(f"  of which at/above tier parity (ctr_vs_tier >= 1.0): "
      f"{int((flagged['ctr_vs_tier']>=1.0).sum()):,}")
print(f"  their share of the top 20: "
      f"{vis.sort_values('baseline_score',ascending=False).head(20)['ctr_vs_tier'].ge(1.0).mean():.0%}")

# Leakage guard: assert none of the label-source columns enter the rule
banned = {"trend_direction", "trend_pct", "is_declining_label"}
rule_inputs = {"ctr", "impressions_90d"}
print("\nLeakage guard — label-source columns used as rule inputs:",
      sorted(banned & rule_inputs) or "none — clean")

# Dummy floor for context
print(f"Dummy floor (majority class): {max(y.mean(), 1-y.mean()):.3f}")


Flagged rows: 9,759
  of which at/above tier parity (ctr_vs_tier >= 1.0): 3,863
  their share of the top 20: 30%

Leakage guard — label-source columns used as rule inputs: none — clean
Dummy floor (majority class): 0.807


## Self-check

- [x] Every section filled — markdown thinking **and** the code that backs it (sections 1–4)
- [x] Notebook runs top to bottom with no errors (executed and saved with outputs)
- [x] Two signal verdicts with visible bucket tables + `n`, at least one flag-linked (both are)
- [x] One rule: a score, one reason code (`low_ctr_high_volume`), one action label (`review_title_meta`)
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [x] Ten reviewed rows, each with "what would make it wrong"
- [x] No future-window or label-derived inputs (fixed `ctr<0.5`, not the tier-relative label)
- [x] No client names, URLs, or private queries — pseudonymized IDs and aggregates only
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`

**Lane locked:** Lane 4 — CTR / Engagement Opportunity Scoring.
**Honest bar for Week 5:** baseline precision@20 = 0.30, @50 = 0.22 (base rate 0.19).
